In [ ]:
import numpy as np

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1','E2','E3'],
    'Hand': ['E4','E5','E6','E7'],
    'Forearm': ['E8','E9','E10','E11','E12','E13'],
    'Arm': ['E14','E15','E16','E17','E18','E19','E20']
}

In [ ]:
ELECTRODE_TO_REGION = {e: r for r, es in REGION_TO_ELECTRODES.items() for e in es}
REGION_TO_LABEL = {'Middle_Finger': 0, 'Hand': 1, 'Forearm': 2, 'Arm': 3}
LABEL_TO_REGION = {v: k for k, v in REGION_TO_LABEL.items()}

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path(__file__).resolve().parent if '__file__' in globals() else Path().resolve()
BIDS_ROOT = NOTEBOOK_DIR.parent.parent / "data" / "BIDS-somatosensory" / "BIDS-somatosensory"
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"
subjects = ["sub-p0002"]
session, task, space = "ses-01", "task-S1Map", "MNI152NLin2009cAsym"
n_runs = 4
HRF_DELAY, WINDOW = 6.0, 1

In [ ]:
import json

bold_json = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_bold.json"
with open(bold_json, 'r', encoding='utf-8') as f:
    tr = float(json.load(f)["RepetitionTime"])

In [ ]:
from nilearn.image import load_img, index_img, new_img_like, resample_to_img
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.maskers import NiftiMasker

first_run_path = DERIVATIVES / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii"
first_run_img = load_img(str(first_run_path))
ref_img = index_img(first_run_img, 0)

atlas = fetch_atlas_destrieux_2009()
#select only the left postcentral gyrus (right side stimulation)
s1_indices = [i for i, lab in enumerate(atlas.labels) if 'L G_postcentral' in str(lab) and i != 0]
atlas_img = load_img(atlas.maps)
mask_data = np.isin(atlas_img.get_fdata(), s1_indices).astype('uint8')
s1_mask = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation='nearest')
masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
masker.fit(first_run_img)

In [ ]:
import pandas as pd

all_events = []
for run in range(1, n_runs + 1):
    events_path = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-{run}_events.tsv"
    df = pd.read_csv(events_path, sep='\t')
    df['run'] = run
    all_events.append(df)
events_df = pd.concat(all_events, ignore_index=True)
stim_events = events_df[~events_df['trial_type'].isin(['Baseline', 'Jitter'])].copy()
stim_events['region'] = stim_events['trial_type'].map(ELECTRODE_TO_REGION)

X_list, y_list = [], []
for run in range(1, n_runs + 1):
    bold_path = DERIVATIVES / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii"
    bold_img = load_img(str(bold_path))
    run_events = stim_events[stim_events['run'] == run]
    for _, event in run_events.iterrows():
        target_time = event["onset"] + HRF_DELAY
        vol_indx = int(np.round(target_time / tr))
        vol_indices = [vol_indx - WINDOW, vol_indx, vol_indx + WINDOW]
        if all(0 <= v < bold_img.shape[3] for v in vol_indices):
            vol_data_list = [masker.transform(index_img(bold_img, v)) for v in vol_indices]
            vol_data_list = [vd[0] if len(vd.shape) == 2 else vd for vd in vol_data_list]
            X_list.append(np.mean(vol_data_list, axis=0))
            y_list.append(event["region"])

X = np.array(X_list)
y_encoded = np.array([REGION_TO_LABEL[r] for r in y_list])
print(f"X shape: {X.shape}, y shape: {y_encoded.shape}")

In [ ]:
RESULTS_DIR = Path("results/4_Classes_ANN")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA   

best_h1, best_h2, best_dr, best_pca, num_classes = 128, 32, 0.6, 50, 4
scaler_all = StandardScaler()
X_scaled_all = scaler_all.fit_transform(X)
pca_all = PCA(n_components=best_pca, random_state=RANDOM_SEED)
X_pca_all = pca_all.fit_transform(X_scaled_all)

In [ ]:
from torch.nn import Module, Linear, ReLU, Sequential, Dropout, BatchNorm1d
import torch

#Two-hidden-layer fully connected (50->128->32->4)
class SomatotopicANN(Module):
    def __init__(self, input_size, hidden1, hidden2, num_classes, dropout_rate):
        super().__init__()
        layers = [Linear(input_size, hidden1), BatchNorm1d(hidden1), ReLU(), Dropout(dropout_rate)]
        layers += [Linear(hidden1, hidden2), BatchNorm1d(hidden2), ReLU(), Dropout(dropout_rate),
                       Linear(hidden2, num_classes)]
        
        self.network = Sequential(*layers)
    def forward(self, x):
        return self.network(x)

In [ ]:
model = SomatotopicANN(best_pca, best_h1, best_h2, num_classes, best_dr)
model.load_state_dict(torch.load(RESULTS_DIR / "models"/ "model_h128-32_wd0.5_dr0.6_pca50.pt", weights_only=True))
model.eval()

In [ ]:
n_voxels = X.shape[1]
saliency_voxel = np.zeros((num_classes, n_voxels))
X_pca_tensor = torch.FloatTensor(X_pca_all).requires_grad_(True)

In [ ]:
#gradient measures how sensitive class 'c' prediction is to PCA component. 
#projects sensitivity into individual voxels -> which voxels the network uses for each region

for c in range(num_classes):
    model.zero_grad()
    if X_pca_tensor.grad is not None:
        X_pca_tensor.grad.zero_()

    outputs = model(X_pca_tensor)           # (N, 4)
    class_score = outputs[:, c].sum()     
    class_score.backward()
    grad_pca = X_pca_tensor.grad.detach().numpy() # (N, 50)

    # Average absolute gradient across samples of c class
    class_mask = y_encoded == c
    mean_grad_pca = np.mean(np.abs(grad_pca[class_mask]), axis=0)  # (50,)
    saliency_pca_to_voxel = mean_grad_pca @ pca_all.components_  # (n_voxels,)
    saliency_voxel[c] = saliency_pca_to_voxel / scaler_all.scale_

In [ ]:
from nilearn import plotting
from IPython.display import display

for c in range(num_classes):
    # Convert voxel vector back to a NIfTI image via the masker
    saliency_img = masker.inverse_transform(saliency_voxel[c].reshape(1, -1))

    view = plotting.view_img(
        saliency_img,
        title=f'Saliency: {LABEL_TO_REGION[c]} (Class {c})',
        threshold=np.percentile(saliency_voxel[c],75),
        colorbar=True,
        symmetric_cmap=False,
    )
    display(view)


In [ ]:
# For each voxel, assign it to the class with highest saliency

TOP_PERCENTILE = 50
winner_map = np.argmax(saliency_voxel, axis=0)
max_saliency = np.amax(saliency_voxel, axis=0)
threshold = np.percentile(max_saliency, TOP_PERCENTILE)
winner_values = np.where(max_saliency >= threshold, winner_map + 1, 0).astype(np.float64)
winner_img = masker.inverse_transform(winner_values.reshape(1, -1))

view = plotting.view_img_on_surf(
    winner_img,
    surf_mesh='fsaverage',
    title="Somatotopic Map (1=Middle_Finger, 2=Hand, 3=Forearm, 4=Arm)",
    symmetric_cmap=False,
    vmin=1,
    vmax=4,
    threshold=1,
    vol_to_surf_kwargs={"interpolation":"nearest_most_frequent"}
)
view